# Patient 360 with DBT

**Author** Jessica Faulk

<i> Creating a event based log of all patient activties as fact that that can be utilized to understand what events are happening when for a patient and allow us to understand outcomes based on previous events, how the jounry might affect readmission or payment behavior, or even some predictions. </i>


---

For this exercise I want to try to create a patient 360 journey that can be used to track events related to a patient using electronic healthcare records. Since PHI is a concern here I will actually utilize a publically avaiable dataset called Synetha that is provide free of cost by the MITRE organization.

Synthea is a mock EHR record set that can be access on many platforms (such as Big Query) but can also be generated. I will start with the simple version found in Big Query but I do plan to include the claim line item details along with seed files to generate a more robust version at a later date.

If you would like to check out Synthea you can find the [link here](https://synthea.mitre.org/).

In [1]:
# 1. Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()

# 2. Mount Google Drive to save your dbt project files
from google.colab import drive
drive.mount('/content/drive')

# 3. Install dbt and the BigQuery adapter
!pip install dbt-bigquery --quiet

print("Environment Ready: BigQuery Authenticated & Drive Mounted.")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.8/172.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.5/175.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.9/144.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.7/442.7 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.5/178.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

# Initialize DBT

Everything is mounted up and ready to go. We will move on to the next phase which is to create a place for the DBT to live.

In [2]:
# 1. Ensure we are in the base directory
%cd /content/drive/MyDrive/dbt_patient_journey

# 2. Run init without the problematic flags
# It will ask: "Which database would you like to use?" Type '1' for bigquery (or the number shown for bigquery).
# It will ask for your project name. Type: patient_journey_360
!dbt init patient_journey_360

/content/drive/MyDrive/dbt_patient_journey
16:32:23  Running with dbt=1.11.4
16:32:23  Creating dbt configuration folder at /root/.dbt
16:32:23  A project called patient_journey_360 already exists here.


At this point, we will use the change directory command to move into the correct folder and then use this new folder to make a profile

In [3]:
import os

# Move into the project
%cd /content/drive/MyDrive/dbt_patient_journey/patient_journey_360

# Create the .dbt directory in your Colab HOME (not Drive)
!mkdir -p ~/.dbt

# Write the profile file
profile_content = """
patient_journey_360:
  outputs:
    dev:
      type: bigquery
      method: oauth
      project: syntheapatientrecords
      dataset: patient_journey_360
      threads: 1
      timeout_seconds: 300
      location: US
      priority: interactive
  target: dev
"""

with open(os.path.expanduser("~/.dbt/profiles.yml"), "w") as f:
    f.write(profile_content)

print("✅ profiles.yml written to ~/.dbt/profiles.yml")

/content/drive/MyDrive/dbt_patient_journey/patient_journey_360
✅ profiles.yml written to ~/.dbt/profiles.yml


In [4]:
# Check your connection - this should give us all green checkmarks!
!dbt debug

16:32:31  Running with dbt=1.11.4
16:32:31  dbt version: 1.11.4
16:32:31  python version: 3.12.12
16:32:31  python path: /usr/bin/python3
16:32:31  os info: Linux-6.6.105+-x86_64-with-glibc2.35
16:33:19  Using profiles dir at /root/.dbt
16:33:19  Using profiles.yml file at /root/.dbt/profiles.yml
16:33:19  Using dbt_project.yml file at /content/drive/MyDrive/dbt_patient_journey/patient_journey_360/dbt_project.yml
16:33:19  adapter type: bigquery
16:33:19  adapter version: 1.11.0
16:33:19  Configuration:
16:33:19    profiles.yml file [OK found and valid]
16:33:19    dbt_project.yml file [OK found and valid]
16:33:19  Required dependencies:
16:33:19   - git [OK found]

16:33:19  Connection:
16:33:19    method: oauth
16:33:19    database: syntheapatientrecords
16:33:19    execution_project: syntheapatientrecords
16:33:19    schema: patient_journey_360
16:33:19    location: US
16:33:19    priority: interactive
16:33:19    maximum_bytes_billed: None
16:33:19    impersonate_service_account: 

### Updating the config file

I want to make sure that the config file contains certain key changes. Doing them here ensures that I am in the correct directory to make this simpler but technically you can do this at any point.

By adding a variable called limit size, I can ensure that my base number of records is limited to 100, but make it easier to change the entire set up later down the line since variables can be changed upon run.

In [24]:
## Here I will update the config files to allow for a variable called limit_size that defaults to 100
import yaml

# Load existing config
with open("dbt_project.yml", "r") as f:
    config = yaml.safe_load(f)

# Add the global variable
config['vars'] = {
    'limit_size': 10
}

# Save it back
with open("dbt_project.yml", "w") as f:
    yaml.dump(config, f)

print("✅ Added 'limit_size' variable to dbt_project.yml")

✅ Added 'limit_size' variable to dbt_project.yml


## Creation of architecture

Following the rest of this we will want to start to define the architecture of the dataset to handle the incoming data. In this process we do not query data, we define what it is... or in this case 'declare' it just like a variable.

1. Create a staging area - I will just call this 'Synthea' inside the 'staging' folder

2. Create a .yml file that defines the entire dataset at a high level with names and descriptions.

### Define Sources

Initial raw data from this dataset comes in as a JSON format that is nested. This can be queried but is not ideal for any kind of analysis. Our first step is the use the staging layer to unpack all this data.

In [6]:
# Create the directory for your staging models if it doesn't exist
!mkdir -p models/staging/synthea

# Write the source definition
sources_yaml = """
version: 2

sources:
  - name: fhir_synthea
    database: bigquery-public-data
    schema: fhir_synthea
    tables:
      - name: patient
        description: "Demographic data for the synthetic population."
      - name: encounter
        description: "Clinical visits and interactions."
      - name: observation
        description: "Vitals, lab results, and clinical measurements."
      - name: condition
        description: "Diagnoses and health problems."
      - name: procedure
        description: "Surgeries, exams, and other medical actions."
      - name: claim
        description: "Claim data"
      - name: explanation_of_benefit
        description: "EOB responses from the payor"

"""

with open("models/staging/synthea/sources.yml", "w") as f:
    f.write(sources_yaml)

print("✅ sources.yml created in models/staging/synthea/")

✅ sources.yml created in models/staging/synthea/


## Variables and Ephemeral Models

I want make sure that I am not pulling all records from the dataset as it is rather huge. However, if I pull 100 records from each object then I am likely to end up with a multitude of orphan records as the 100 take from each object will only be the top 100 results.

To handle this I want to make sure that anything coming after the intial patient stage can reference back to the original 100 patient ids. This can be handled with an ephemeral model. The ephemeral model is basically like a SQL injection, but in a good way. Think of it like a resuable query.

Addtionally, we want to think bigger here. In the short term setting my limits to 100 patients seems good but what happens if I want to dynmaically expand the size to 1000 or even 10,000 patients? In this case we can actually define a vaiable to handle this.

In [7]:
# Create the cohort model file - aka ephemeral model
int_cohort_sql = """
{{ config(materialized='ephemeral') }}

select
    id as patient_id
from {{ source('fhir_synthea', 'patient') }}
limit {{ var('limit_size') }}
"""

with open("models/staging/synthea/int_cohort.sql", "w") as f:
    f.write(int_cohort_sql)

print("✅ Ephemeral 'int_cohort' created with variable-based limit.")

✅ Ephemeral 'int_cohort' created with variable-based limit.


# Staging Layers

## Patients Staging

This is the area where we will unpack the data from the original sources that we defined in the sources layer above.

We are also going to do a couple of things here.

1. Limit to only 100 records because I don't want to drive up costs during development.

2. We will reference the original list of patientIDs to make sure that we do not pull a bunch of orphan records later on

Additionally, we will add some metadata to reduce the later effort if I need to reload tables again. For this exercise I will add two new items.

1. The current timestamp that the information was run - CURRENT_TIMESTAMP


Now I can add in to make this incremental but since I plan to use a more static dataset, it's not really required at this time. However, it would make sense to do this in other tables such as encounters to reduce the amount of data that would be rewritten for any subsequent encounters.

In [99]:
stg_patients_sql = """

{{ config(materialized='table') }}

select
    id as patient_id,
    safe.parse_timestamp('%Y-%m-%dT%H:%M:%SZ',birthDate) as birth_date,
    safe.parse_timestamp('%Y-%m-%dT%H:%M:%SZ',deceased.dateTime) as death_date,
    gender,
    -- Extracting specific identifiers if needed
    -- Standardizing address (Synthea uses nested arrays)
    address[SAFE_OFFSET(0)].city as city,
    address[SAFE_OFFSET(0)].state as state,
    address[SAFE_OFFSET(0)].postalCode as postal_code,
    -- Metadata below
    current_timestamp() as _dbt_updated_at
from {{ source('fhir_synthea', 'patient') }}
limit {{ var('limit_size') }}

"""

with open("models/staging/synthea/stg_patients.sql", "w") as f:
    f.write(stg_patients_sql)

print("✅ stg_patients.sql model created.")

✅ stg_patients.sql model created.


 ### Running DBT to actually create out models

 Now that we have the source and patient objects defined, we can run DBT to construct these objects for us in the BigQuery project I have create for myself.


 In my intial run of DBT, I was returned an error on a column name that did not match. Taking a look at the data sources again, I found that the date of death was actually collected in a nested record of DateTime within the Deceased field.

 To fix that, I adjusted the script above and overwrote the original stg_patient.yml file and then re-ran DBT.

## Staging Everything Else

The nice thing about using DBT is that we can actually combine multiple tasks into one run if we want. This may not be a great choice to have one large complex script for the staging layer.

It won't really matter to DBT since each .SQL file is distinct and DBT will use a DAG process to run the scripts in order. The nice thing about this is that if something fails upstream, DBT will not run any downstream models that were affected by this error.

This also gives us an advantage that we can utilize later for QA checking and error messages.

Additionally, here is where I want to set up the incremental refresh as I only want to add new events to the end log as they occur. By setting the incremental arguements here I can easily only pull NEW events related to the patient cohort.

Since we want to capture both NEW and UPDATED records, we will need to specify the key.


In [102]:
stg_encounters_sql = """

{{
    config(
        materialized='incremental',
        unique_key='encounter_id',
        incremental_strategy='merge'
    )
}}


select
    id as encounter_id,
    subject.patientId as patient_id,
    class.code as encounter_class,
    -- FIX: Casting Strings to Timestamps for BigQuery math
    safe.parse_timestamp('%Y-%m-%dT%H:%M:%SZ', period.start) as start_timestamp,
    safe.parse_timestamp('%Y-%m-%dT%H:%M:%SZ', period.end) as end_timestamp,
    -- Metric: Length of Stay in Minutes
    timestamp_diff(
        safe.parse_timestamp('%Y-%m-%dT%H:%M:%SZ', period.end),
        safe.parse_timestamp('%Y-%m-%dT%H:%M:%SZ', period.start),
        MINUTE) as duration_minutes,
    -- Metadata below
    current_timestamp() as _dbt_updated_at
from {{ source('fhir_synthea', 'encounter') }} e

-- only records from the cohort are returned
where e.subject.patientId in (select patient_id from {{ ref('int_cohort') }})

{% if is_incremental() %}
  -- This filter only applies on incremental runs
  -- It ensures we only process data newer than the max date already in the table
  and safe.parse_timestamp('%Y-%m-%dT%H:%M:%SZ', period.start) > (select max(safe.parse_timestamp('%Y-%m-%dT%H:%M:%SZ', period.start)) from {{ this }})
{% endif %}

"""

with open("models/staging/synthea/stg_encounters.sql", "w") as f:
    f.write(stg_encounters_sql)

print("✅ stg_encounters.sql created.")


stg_observations_sql = """

{{
    config(
        materialized='incremental',
        unique_key='observation_id',
        incremental_strategy='merge'
    )
}}

select
    id as observation_id,
    subject.patientId as patient_id,
    context.encounterId as encounter_id,
    -- Clinical coding (LOINC)
    code.coding[SAFE_OFFSET(0)].code as loinc_code,
    code.coding[SAFE_OFFSET(0)].display as observation_type,
    -- Results (Coalescing numeric and string values)
    coalesce(
        cast(value.quantity.value as string),
        value.codeableConcept.text,
        cast(value.string as string)
    ) as result_value,
    value.quantity.unit as unit,
    effective.dateTime as effective_at,
    -- Metadata below
    current_timestamp() as _dbt_updated_at
from {{ source('fhir_synthea', 'observation') }} o
where o.subject.patientId in (select patient_id from {{ ref('int_cohort') }})


{% if is_incremental() %}
  -- This filter only applies on incremental runs
  -- It ensures we only process data newer than the max date already in the table
  and effective.dateTime > (select max(effective.dateTime) from {{ this }})
{% endif %}

"""

with open("models/staging/synthea/stg_observations.sql", "w") as f:
    f.write(stg_observations_sql)

print("✅ stg_observations.sql created.")


stg_conditions_sql = """

{{
    config(
        materialized='incremental',
        unique_key='condition_id',
        incremental_strategy='merge'
    )
}}


select
    id as condition_id,
    subject.patientId as patient_id,
    context.encounterId as encounter_id,
    -- Clinical coding (SNOMED)
    code.coding[SAFE_OFFSET(0)].code as snomed_code,
    code.coding[SAFE_OFFSET(0)].display as condition_description,
    onset.dateTime as diagnosed_at,
    abatement.dateTime as resolved_at,
    -- Metadata below
    current_timestamp() as _dbt_updated_at
from {{ source('fhir_synthea', 'condition') }} c
where c.subject.patientId in (select patient_id from {{ ref('int_cohort') }})


{% if is_incremental() %}
  -- This filter only applies on incremental runs
  -- It ensures we only process data newer than the max date already in the table
  and onset.dateTime > (select max(onset.dateTime) from {{ this }})
{% endif %}


"""

with open("models/staging/synthea/stg_conditions.sql", "w") as f:
    f.write(stg_conditions_sql)

print("✅ stg_conditions.sql created.")

# Staging for the Billing Request (Claim)
stg_claims_sql = """

{{
    config(
        materialized='incremental',
        unique_key='claim_id',
        incremental_strategy='merge'
    )
}}

select
    id as claim_id,
    patient.patientId as patient_id,
    billablePeriod.start as bill_date,
    total.value as claimed_amount,
    status as claim_status,
    meta.lastUpdated as updated_at
from {{ source('fhir_synthea', 'claim') }}
where patient.patientId in (select patient_id from {{ ref('int_cohort') }})


{% if is_incremental() %}
  -- This filter only applies on incremental runs
  -- It ensures we only process data newer than the max date already in the table
  and meta.lastUpdated > (select max(meta.lastUpdated) from {{ this }})
{% endif %}

"""

with open("models/staging/synthea/stg_claims.sql", "w") as f:
    f.write(stg_claims_sql)

print("✅ Successfully separated Claim (Request).")

# Staging for the Final Payment (EOB)
stg_eob_sql = """

{{
    config(
        materialized='incremental',
        unique_key='eob_id',
        incremental_strategy='merge'
    )
}}


select
    id as eob_id,
    claim.claimId as related_claim_id,
    patient.patientId as patient_id,
    totalCost.value as total_adjudicated_cost,
    item[SAFE_OFFSET(0)].serviced.period.start as service_date,
    outcome as adjudication_outcome,
    meta.lastUpdated as updated_at
from {{ source('fhir_synthea', 'explanation_of_benefit') }}
where patient.patientId in (select patient_id from {{ ref('int_cohort') }})


{% if is_incremental() %}
  -- This filter only applies on incremental runs
  -- It ensures we only process data newer than the max date already in the table
  and meta.lastUpdated > (select max(meta.lastUpdated) from {{ this }})
{% endif %}

"""

with open("models/staging/synthea/stg_eob.sql", "w") as f:
    f.write(stg_eob_sql)

print("✅ Successfully separated EOB (Result).")

✅ stg_encounters.sql created.
✅ stg_observations.sql created.
✅ stg_conditions.sql created.
✅ Successfully separated Claim (Request).
✅ Successfully separated EOB (Result).


# Making the Market Happen

Now that we have data staged we can move into making the marts. We will need to make a file to contain the marts.

In [10]:
!mkdir -p models/marts

## Create the Fact Table for our mart

We want to stack up the encounters and the observations and align them into a unified scheme to generate our longitudinal model.

In this step we can also add in some calculations that can return the number of days since the previous event

In [33]:
fct_patient_journey_sql = """

{{
    config(
        materialized='incremental',
        unique_key=['event_id', 'event_type'],
        incremental_strategy='merge'
    )
}}


-- We stack all clinical events into one timeline
with events as (
    -- 1. Encounters
    select
        encounter_id as event_id,
        patient_id,
        start_timestamp as event_date,
        'Encounter' as event_type,
        encounter_class as event_details
    from {{ ref('stg_encounters') }}

    union all

    -- 2. Observations
    select
        observation_id as event_id,
        patient_id,
        cast(effective_at as timestamp) as event_date,
        'Observation' as event_type,
        observation_type || ': ' || result_value as event_details
    from {{ ref('stg_observations') }}

    union all

    -- 3. Conditions
    select
        condition_id as event_id,
        patient_id,
        cast(diagnosed_at as timestamp) as event_date,
        'Condition' as event_type,
        condition_description as event_details
        from {{ ref('stg_conditions') }}

    union all

    -- 4. claims
    select
        claim_id as event_id,
        patient_id,
        cast(bill_date as timestamp) as event_date,
        'Claim' as event_type,
        'Claim' as event_details
        from {{ ref('stg_claims') }}

    union all

    -- 5. EOBs
    select
        eob_id as event_id,
        patient_id,
        cast(service_date as timestamp) as event_date,
        'EOB' as event_type,
        'EOB' as event_details
        from {{ ref('stg_eob') }}

),


enriched_events as (
    select
        *,
        -- 1. Sequence within the journey
        row_number() over (partition by patient_id order by event_date) as event_seq,

        -- 2. Time delta from previous event
        timestamp_diff(
            event_date,
            lag(event_date) over (partition by patient_id order by event_date),
            day
        ) as days_since_prev_event,

        -- 3. Running total of events (Acuity)
        count(*) over (partition by patient_id order by event_date rows between unbounded preceding and current row) as cumulative_event_count
    from events
)

select * from enriched_events

{% if is_incremental() %}
  -- This filter only applies on incremental runs
  -- It ensures we only process data newer than the max date already in the table
  WHERE event_date > (select max(event_date) from {{ this }})
{% endif %}



"""

with open("models/marts/fct_patient_journey.sql", "w") as f:
    f.write(fct_patient_journey_sql)

print("✅ fct_patient_journey.sql created in models/marts/")

✅ fct_patient_journey.sql created in models/marts/


In [111]:
!dbt run

22:48:32  Running with dbt=1.11.4
22:49:01  Registered adapter: bigquery=1.11.0
22:49:02  Found 9 models, 1 seed, 8 data tests, 7 sources, 652 macros
22:49:02  
22:49:02  Concurrency: 1 threads (target='dev')
22:49:02  
22:49:03  1 of 8 START sql table model patient_journey_360.stg_patients .................. [RUN]
22:49:06  1 of 8 OK created sql table model patient_journey_360.stg_patients ............. [CREATE TABLE (10.0 rows, 104.7 MiB processed) in 2.98s]
22:49:06  2 of 8 START sql incremental model patient_journey_360.stg_claims .............. [RUN]
22:49:10  2 of 8 OK created sql incremental model patient_journey_360.stg_claims ......... [MERGE (0.0 rows, 5.8 GiB processed) in 3.96s]
22:49:10  3 of 8 START sql incremental model patient_journey_360.stg_conditions .......... [RUN]
22:49:23  3 of 8 OK created sql incremental model patient_journey_360.stg_conditions ..... [MERGE (0.0 rows, 1.6 GiB processed) in 12.66s]
22:49:23  4 of 8 START sql incremental model patient_journey_360

**SUCCESS**

The model runs and we have gotten it up to the fact table.

Let's move on to some dimension tables. From this point on, I will try to be specific on what parts I am running as opposed to re-running all of DBT each time which will quickly chew through your data.

## Dimensional Tables

Next we want to create some dimensional tables that can allow us slice and dice our data with ease. These may look very similar to our staging tables but we actually want to leave the staging tables as raw as possible with minimal transformations such as conversions to correct datatypes. Otherwise, we are going to leave the heavy logic to the dimensional tables.

In the dimensional tables, we may want to utilize groupings, mappings, and other ways to clean and standardized data. In order to keep scripts clean and logic consolidated, we will utilize some seed files.

<b><i> What is a seed file? </i></b>

Seed files are usually smaller flat files that contain mappings that can be called over and over again. An example here might be grouping together different response codes from the EOB. Instead of hard coding the logic for what is considered a non-covered service response code within the logic of the script, we will generate a seed file that has all response codes possible and a grouping category.

The beauty of this approach is that it is much easier to add new codes, update groupings, or add new logic. You can even add more columns to the seed file to specify different grouping logic based on the business use case. One team may not want to group a certain code at all. You can do that with columns that can account for that variation.


Our next step will be to create a directory to contain our dimensions tables

In [35]:
!mkdir -p models/marts/dimensions

Once we have a place to put our new dimensions within the mart, we can work on creating the SQL scripts that will build our dimesnion tables.

We don't want to use views here since the dimensions are less prone to changes, so we will make these tables but also run them as incremental.

I also want to bring in some utilities that can make this process easier and provides some more functionality just beyond core DBT.

Let's bring in DB_utils, a package that brings in a few more utility functions.

In [51]:
import os

# Define the content for packages.yml
packages_yml_content = """
packages:
  - package: dbt-labs/dbt_utils
    version: 1.1.1
"""

# Write the content to packages.yml in the current directory - you will want yours to go in the same level as the main project folder
with open("packages.yml", "w") as f:
    f.write(packages_yml_content)

print("✅ packages.yml created with dbt_utils package definition.")

✅ packages.yml created with dbt_utils package definition.


In [52]:
# The seeds directory already exists so I just want to add a file to this.
# In this case we are just going to remap the gender values to something more standard

import pandas as pd

df = pd.DataFrame({
    "gender": ["male", "female", "other"],
    "gender_label": ["M", "F", "O"]
})

df

df.to_csv("seeds/gender_lookup.csv", index=False)

Well, we have created the file in the right place, but we haven't actually told the model to build this asset within our database. To do that we will need to add a seeds section to our dbt_project.yml file.

I will add it in the cell below.

In [71]:
## Updating the dbt_project file here


import yaml

# Add the seeds configuration to the config dictionary
config['seeds'] = {
        '+materialized': 'seed',
        '+gender_lookup': {

            }
        }


# Save the updated config back to dbt_project.yml
with open("dbt_project.yml", "w") as f:
    yaml.dump(config, f)

print("✅ Added seeds configuration to dbt_project.yml")

✅ Added seeds configuration to dbt_project.yml


In [93]:
print("Running dbt seed...")
!dbt seed

Running dbt seed...
22:07:29  Running with dbt=1.11.4
22:07:59  Registered adapter: bigquery=1.11.0
22:07:59  Unable to do partial parsing because a project config has changed
22:08:01  Found 9 models, 1 seed, 8 data tests, 7 sources, 652 macros
22:08:01  
22:08:01  Concurrency: 1 threads (target='dev')
22:08:01  
22:08:03  1 of 1 START seed file patient_journey_360.gender_lookup ....................... [RUN]
22:08:19  1 of 1 OK loaded seed file patient_journey_360.gender_lookup ................... [INSERT 3 in 16.22s]
22:08:19  
22:08:19  Finished running 1 seed in 0 hours 0 minutes and 17.87 seconds (17.87s).
22:08:19  
22:08:19  Completed successfully
22:08:19  
22:08:19  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 TOTAL=1


 Now that we have added the db util package, we can move on to creating the first dim table for patient.


We want to do a few things here:

1. configure it to another incremental view to make sure we aren't loading a ton of data over and over again.

2. Set the unique key for the configuration as well.

3. Calculate the patients age here based on the birthday

4. Create a step to remove any possible duplicate rows

5. Assign a surrogate key

6. Change the state and the city to UPPERCASE




In [107]:
dim_patients_sql = """

{{ config(
    materialized='incremental',
    unique_key='patient_id'
) }}

with source as (
    select * from {{ ref('stg_patients') }}
),

-- Add engineered fields, renaming, cleanup
clean as (
    select
        patient_id,
        CAST(birth_date as DATE) as birth_date,
        CAST(death_date as DATE) as death_date,
        gender,
        UPPER(city) as city,
        UPPER(state) as state,
        _dbt_updated_at,

        -- Derived field
        date_diff(current_date(), CAST(birth_date as DATE), day) / 365.25 as age
    from source
),

-- Optional deduplication in case upstream sends repeated rows
latest as (
    select *
    from clean
    qualify row_number() over (partition by patient_id order by birth_date desc) = 1
),

gender_join as (
    select
        l.patient_id,
        l.birth_date,
        l.death_date,
        l.gender,
        l.city,
        l.state,
        l._dbt_updated_at,
        l.age,
        g.gender_label
    from latest l
    left join {{ ref('gender_lookup') }} g
        on l.gender = g.gender
),

final as (
    select
        {{ dbt_utils.generate_surrogate_key(['patient_id']) }} as patient_sk,
        *
    from gender_join
)

select * from final

{% if is_incremental() %}

-- Only update patients that have changed **OR** are new
where patient_id in (
    select patient_id
    from {{ ref('stg_patients') }}
)

{% endif %}

"""

with open("models/marts/dimensions/dim_patient.sql", "w") as f:
    f.write(dim_patients_sql)

print("✅ dim_patients.sql model created.")


✅ dim_patients.sql model created.


 With the dimensional table added we can continue this process for other staged tables as well. However, I want to move forward and work on adding in some quality tests within the project.

## Quality Testing

DBT offers an easy way to implement testing across your assests that can help keep data fresh and accurate.

I will add some simple quality testing to my current project in order to make sure that my data is not getting duplicated within the data base.

We will need to switch gears and work with our old friend the YML file instead of creating SQL scripts.

Again, YML files act more like a set of map directions with instructions at each location. By doing this programatically, we can reference what tests should be performed where. If we want to change what test is run, we change the YML file as opposed to directly altering the database.

These tests will run when DBT is triggered to run. In this project, I have to trigger the run manually but this can also be automated within orchetrators like Airflow to fully automate the process and include triggers to alert users to the issues so they can be handled in a timely manner.

Since I haven't added it so far, I will be add it in now using the command below.

In [ ]:
print("Installing dbt packages...")
!dbt deps

### Dim Patient YML file

This will have not only what tests should be run but we can add in highly valuable metadata here as well such as a description and acceptable values.

We will use the expression based tests that are accessible within the DB_utils package.

In [84]:
# Write the source definition for dim_patient.yml


dim_patient = """
version: 2
models:
- name: dim_patient
  description: Dimension table for all patients
  columns:
  - name: patient_sk
    description: Surrogate key for each patient
    tests:
    - unique
    - not_null
  - name: patient_id
    description: Source system patient ID
    tests:
    - not_null
  - name: gender
    description: Patient gender code
    tests:
    - accepted_values:
        arguments:
          values: ['M', 'F', 'O']
  - name: age
    description: Calculated age in years
    tests:
    - not_null
    - dbt_utils.expression_is_true:
        arguments:
          expression: age >= 0
  tests:
  - dbt_utils.at_least_one_row
  - patient_historical_consistency

"""

with open("models/marts/dimensions/dim_patient.yml", "w") as f:
    f.write(dim_patient)

print("✅ sources.yml created in models/marts/dimensions/")

✅ sources.yml created in models/marts/dimensions/


### Adding in a custom test to your project

Let's say I have a specific test I want to run that is not the standard. I can actually create my own tests using SQL that will be run when I call Tests.

For this example, I am going to create a test script to check to make sure that slowly changing dimensions (SCD Type 1/ SCD Type 2) are never deleted since I am running things with incremental tables.

Since DBT will run in order, we can load the stage data and check against what is currently in the dimensions table. I want to ensure that older patient IDs will not be lost in the update so I can use the staged table as CURRENT and the existing DIM table as the historical, compare the two, and make sure that no old values are lost.


**WARNING**

This process will only work in my current project for PATIENTS because that staging table is NOT incremental. If your staging table is incremental this test will not work correctly.


**Where Do I Put This Script?**

The test scripts should always be placed in the TEST directory within your project.

In [40]:
patient_historical_consistency = """

-- Test: Ensure patient_ids do not disappear between runs
with historical as (
    select patient_id
    from {{ ref('dim_patient') }}
),

current as (
    select distinct patient_id
    from {{ ref('stg_patients') }}
)

select patient_id
from historical
where patient_id not in (select patient_id from current)

/*
for larger tables you can use this instead
select h.patient_id
from {{ ref('dim_patient') }} h
left join {{ ref('stg_patients') }} c
    on h.patient_id = c.patient_id
where c.patient_id is null
 */

"""

with open("tests/patient_historical_consistency.sql", "w") as f:
    f.write(patient_historical_consistency)

print("✅ patient_historical_consistency.sql model created.")


Let's add the new test to the old file. I could overwrite the entire thing again, or we can just add to the existing file.



In [85]:
import yaml

# Load the content of models/marts/dimensions/patient_dim.yml
with open("models/marts/dimensions/dim_patient.yml", "r") as f:
    patient_dim_config = yaml.safe_load(f)

# Let's add table-level tests to the dim_patient model
if patient_dim_config and 'models' in patient_dim_config and len(patient_dim_config['models']) > 0:
    dim_patient_model = patient_dim_config['models'][0]

    # Initialize 'tests' key if it doesn't exist
    if 'tests' not in dim_patient_model:
        dim_patient_model['tests'] = []

    # Add the new table-level tests
    dim_patient_model['tests'].append('dbt_utils.at_least_one_row')
    dim_patient_model['tests'].append('patient_historical_consistency')

# Now we save the modified YAML content back to the file
with open("models/marts/dimensions/dim_patient.yml", "w") as f:
    yaml.dump(patient_dim_config, f, sort_keys=False)

print("✅ Added table-level tests to dim_patient model in models/marts/dimensions/dim_patient.yml")

✅ Added table-level tests to dim_patient model in models/marts/dimensions/dim_patient.yml


## Adding in the new dim table

DBT will let you run parts of the process selectivly, allowing for easy small scale changes and testing without having to run an entire refresh. I will build the dim table directly without having to call the entire program again.

In [108]:
!dbt run --select dim_patient

22:30:33  Running with dbt=1.11.4
22:31:02  Registered adapter: bigquery=1.11.0
22:31:03  Found 9 models, 1 seed, 8 data tests, 7 sources, 652 macros
22:31:03  
22:31:03  Concurrency: 1 threads (target='dev')
22:31:03  
22:31:05  1 of 1 START sql incremental model patient_journey_360.dim_patient ............. [RUN]
22:31:08  1 of 1 OK created sql incremental model patient_journey_360.dim_patient ........ [CREATE TABLE (10.0 rows, 797.0 Bytes processed) in 3.44s]
22:31:08  
22:31:08  Finished running 1 incremental model in 0 hours 0 minutes and 5.04 seconds (5.04s).
22:31:08  
22:31:08  Completed successfully
22:31:08  
22:31:08  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 TOTAL=1


## Running the tests

In [110]:
print("Running dbt tests...")
!dbt test

Running dbt tests...
22:36:20  Running with dbt=1.11.4
22:36:50  Registered adapter: bigquery=1.11.0
22:36:51  Found 9 models, 1 seed, 8 data tests, 7 sources, 652 macros
22:36:51  
22:36:51  Concurrency: 1 threads (target='dev')
22:36:51  
22:36:52  1 of 8 START test accepted_values_dim_patient_gender__M__F__O .................. [RUN]
22:36:54  1 of 8 FAIL 2 accepted_values_dim_patient_gender__M__F__O ...................... [FAIL 2 in 1.72s]
22:36:54  2 of 8 START test dbt_utils_at_least_one_row_dim_patient_ ...................... [RUN]
22:36:54  2 of 8 ERROR dbt_utils_at_least_one_row_dim_patient_ ........................... [ERROR in 0.02s]
22:36:54  3 of 8 START test dbt_utils_expression_is_true_dim_patient_age__age_0 .......... [RUN]
22:36:54  BigQuery adapter: https://console.cloud.google.com/bigquery?project=syntheapatientrecords&j=bq:US:53a247b3-3455-4db7-859e-97850eb3f9bf&page=queryresults
22:36:54  3 of 8 ERROR dbt_utils_expression_is_true_dim_patient_age__age_0 .............

## Taking a Pause here

So that's where this walk through end for now. It takes a long time for me to create the tutorial, look up each reason and write up the process.

In this walkthrough we did the following:

1. Connect our collab notebook up to the synthea data set
2. Staged the base cohort with an ephermeral table and a global variable
3. Staged the data and used incremental refresh for the rest of the data
4. Did some data cleaning by changing strings to datetimes
5. Used our staged data to create a centralized event log of the patient's activity.
6. Created our first DIM table to help slice up our FACT table
7. Utilized a SEED to easily remap values
8. Loaded in DB_utils to run some standard tests and create a surrogate key on on DIM table
9. Created our own custom test to run
10. Ran our tests against the warehouse


For next time we have some work to do on transforming the dates in the data from strings into DATEs and DATETIMEs that we can utilize better. Additionally, it seems like our test for the gender labels is being run against the original gender column. I am thinking that we will just remove the gender column entirely as we want our standardized format.


